<a href="https://colab.research.google.com/github/Adhiaris/Practical-Statistics-for-Data-Scientist-Books/blob/main/Practical_Statistics_Chapter6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 6: Statistical Machine Learning

This chapter covers four core methods for tabular data:
1. **K-Nearest Neighbors (KNN)** — classify by similarity.
2. **Decision Trees (CART)** — interpretable if-then rules.
3. **Random Forests** — bagged, decorrelated trees for lower variance.
4. **Boosting (XGBoost)** — sequential error correction for high accuracy.

All examples use LendingClub loan data to predict default vs. paid-off.


## Setup and Imports


In [ ]:
!pip install dmba

In [ ]:
import math
import os
import random
from pathlib import Path
from collections import defaultdict
from itertools import product

import pandas as pd
import numpy as np

from sklearn import preprocessing
from sklearn.neighbors import KNeighborsClassifier
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from dmba import plotDecisionTree, textDecisionTree

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import warnings; warnings.filterwarnings('ignore')

print("All libraries loaded successfully.")
print(f"scikit-learn: {__import__('sklearn').__version__}")
print(f"XGBoost: {__import__('xgboost').__version__}")


We use scikit-learn (KNN, trees, random forests), XGBoost (boosting), and dmba (tree visualization). Install missing packages with `!pip install dmba xgboost`.


### Load Data

Three data subsets:
- `loan200` — 200 records for the introductory KNN example.
- `loan3000` — 3,000 records for tree demonstrations.
- `loan_data` — ~45,000 records for random forest and boosting.


In [ ]:
import os
if not os.path.exists('psfds'):
    !git clone --depth 1 https://github.com/gedeck/practical-statistics-for-data-scientists.git psfds

DATA = 'psfds/data'
LOAN200_CSV = f'{DATA}/loan200.csv'
LOAN3000_CSV = f'{DATA}/loan3000.csv'
LOAN_DATA_CSV = f'{DATA}/loan_data.csv.gz'

loan200 = pd.read_csv(f'{DATA}/loan200.csv')
print(f"loan200:    {loan200.shape[0]} records x {loan200.shape[1]} columns")
print(f"Columns: {list(loan200.columns)}")
print(loan200.head(3))

`loan200` is small enough to visualize KNN neighborhoods in 2D. Features: `dti` (debt-to-income) and `payment_inc_ratio`.


## K-Nearest Neighbors

KNN predicts a new record's class by majority vote among the $K$ most similar training records. There is no model to fit — all computation happens at prediction time.


In [ ]:
predictors = ['payment_inc_ratio', 'dti']
outcome = 'outcome'

newloan = loan200.loc[0:0, predictors]
X = loan200.loc[1:, predictors]
y = loan200.loc[1:, outcome]

knn = KNeighborsClassifier(n_neighbors=20)
knn.fit(X, y)
knn.predict(newloan)
print("Probability estimates [P(paid off), P(default)]:")
print(knn.predict_proba(newloan))


Among 20 nearest neighbors, 11 paid off and 9 defaulted → $P(\text{paid off})=0.55$. This probability score is more useful than a binary label: a lender can set any cutoff based on risk tolerance.


In [ ]:
nbrs = knn.kneighbors(newloan)
maxDistance = np.max(nbrs[0][0])

fig, ax = plt.subplots(figsize=(5, 5))
sns.scatterplot(x='payment_inc_ratio', y='dti', style='outcome',
                hue='outcome', data=loan200, alpha=0.3, ax=ax)
sns.scatterplot(x='payment_inc_ratio', y='dti', style='outcome',
                hue='outcome',
                data=pd.concat([loan200.loc[0:0, :], loan200.loc[nbrs[1][0] + 1,:]]),
                ax=ax, legend=False)
ellipse = Ellipse(xy=newloan.values[0],
                  width=2 * maxDistance, height=2 * maxDistance,
                  edgecolor='black', fc='None', lw=1)
ax.add_patch(ellipse)
ax.set_xlim(3, 16)
ax.set_ylim(15, 30)

plt.tight_layout()
plt.show()


The ellipse encloses the 20 nearest neighbors. The new loan sits at the center; the prediction is a simple vote count.

**Bias-variance tradeoff:** Small $K$ → high variance (sensitive to noise); large $K$ → high bias (averages too broadly).
$$\text{Total Error} = \text{Bias}^2 + \text{Variance} + \sigma^2$$


### Standardization

KNN uses Euclidean distance, so high-magnitude features dominate. **Standardization** puts all features on the same scale:
$$z = \frac{x - \bar{x}}{s}$$

Without it, `revol_bal` (range: thousands) completely overwhelms `dti` (range: ~40).


In [ ]:
loan_data = pd.read_csv(LOAN_DATA_CSV)
loan_data = loan_data.drop(columns=['Unnamed: 0', 'status'])
loan_data['outcome'] = pd.Categorical(loan_data['outcome'],
                                      categories=['paid off', 'default'],
                                      ordered=True)

predictors = ['payment_inc_ratio', 'dti', 'revol_bal', 'revol_util']
outcome = 'outcome'

newloan = loan_data.loc[0:0, predictors]
print("New loan to predict:")
print(newloan)
X = loan_data.loc[1:, predictors]
y = loan_data.loc[1:, outcome]

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X, y)

nbrs = knn.kneighbors(newloan)
print("\n5 Nearest Neighbors WITHOUT Standardization:")
print(X.iloc[nbrs[1][0], :])


Without scaling, all five nearest neighbors cluster tightly on `revol_bal` — the other three features are effectively ignored.


In [ ]:
newloan = loan_data.loc[0:0, predictors]
X = loan_data.loc[1:, predictors]
y = loan_data.loc[1:, outcome]

scaler = preprocessing.StandardScaler()
scaler.fit(X * 1.0)

X_std = scaler.transform(X * 1.0)
newloan_std = scaler.transform(newloan * 1.0)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_std, y)

nbrs = knn.kneighbors(newloan_std)
print("5 Nearest Neighbors WITH Standardization:")
print(X.iloc[nbrs[1][0], :])


After scaling, all four features contribute proportionally to the distance. Always fit `StandardScaler` on training data only — applying training statistics to test data prevents data leakage.


### KNN as a Feature Engine

KNN's main modern use is generating a `borrower_score` feature — the fraction of $K$ credit-history neighbors who paid off. This local creditworthiness score becomes a powerful predictor in downstream models.


In [ ]:
loan_data = pd.read_csv(LOAN_DATA_CSV)
loan_data = loan_data.drop(columns=['Unnamed: 0', 'status'])
loan_data['outcome'] = pd.Categorical(loan_data['outcome'],
                                      categories=['paid off', 'default'],
                                      ordered=True)

predictors = ['dti', 'revol_bal', 'revol_util', 'open_acc',
              'delinq_2yrs_zero', 'pub_rec_zero']
outcome = 'outcome'

X = loan_data[predictors]
y = loan_data[outcome]

knn = KNeighborsClassifier(n_neighbors=20)
knn.fit(X, y)
plt.scatter(range(len(X)), [bs + random.gauss(0, 0.015) for bs in knn.predict_proba(X)[:,0]],
            alpha=0.1, marker='.')

loan_data['borrower_score'] = knn.predict_proba(X)[:, 0]
print(loan_data['borrower_score'].describe())

plt.xlabel('Record index')
plt.ylabel('borrower_score (jittered)')
plt.title('KNN-Derived Borrower Score Distribution')
plt.tight_layout()
plt.show()


`borrower_score` distills six credit features into a single number in $[0, 1]$: fraction of 20 nearest neighbors who paid off. Scores near 0.95 indicate very creditworthy borrowers; near 0.00 indicate high risk.


## Tree Models

Decision trees create nested if-then rules by recursively partitioning data into purer groups. They are transparent and easy to explain to non-technical stakeholders.


In [ ]:
loan3000 = pd.read_csv(LOAN3000_CSV)

predictors = ['borrower_score', 'payment_inc_ratio']
outcome = 'outcome'

X = loan3000[predictors]
y = loan3000[outcome]

loan_tree = DecisionTreeClassifier(random_state=1, criterion='entropy',
                                   min_impurity_decrease=0.003)
loan_tree.fit(X, y)
plotDecisionTree(loan_tree, feature_names=predictors, class_names=loan_tree.classes_)


The tree uses `borrower_score` at three of five levels — it is the dominant predictor. `payment_inc_ratio` only refines classification in the ambiguous middle range (score 0.375–0.575).

**Three examples:**
- score=0.6, ratio=8 → root: 0.6≥0.575 → **paid off** (1 step)
- score=0.3, ratio=12 → root: no → level 2: 0.3<0.375 → **default** (2 steps)
- score=0.45, ratio=5 → traverses 5 nodes → **default** (score 0.45 < threshold 0.475)


In [ ]:
print(textDecisionTree(loan_tree))


Text representation: each node shows feature index (0=`borrower_score`, 1=`payment_inc_ratio`), threshold, and leaf class probabilities $[P(\text{default}), P(\text{paid off})]$.


### Recursive Partitioning

Each tree split draws an axis-aligned line, creating rectangular decision regions in feature space.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

loan3000.loc[loan3000.outcome=='paid off'].plot(
    x='borrower_score', y='payment_inc_ratio', style='.',
    markerfacecolor='none', markeredgecolor='C1', ax=ax)
loan3000.loc[loan3000.outcome=='default'].plot(
    x='borrower_score', y='payment_inc_ratio', style='o',
    markerfacecolor='none', markeredgecolor='C0', ax=ax)
ax.legend(['paid off', 'default']);
ax.set_xlim(0, 1)
ax.set_ylim(0, 25)
ax.set_xlabel('borrower_score')
ax.set_ylabel('payment_inc_ratio')

x0 = 0.575
x1a = 0.325; y1b = 9.191
y2a = 10.423; x2b = 0.725
ax.plot((x0, x0), (0, 25), color='grey')
ax.plot((x1a, x1a), (0, 25), color='grey')
ax.plot((x0, 1), (y1b, y1b), color='grey')
ax.plot((x1a, x0), (y2a, y2a), color='grey')
ax.plot((x2b, x2b), (0, y1b), color='grey')

labels = [('default', (x1a / 2, 25 / 2)),
          ('default', ((x0 + x1a) / 2, (25 + y2a) / 2)),
          ('paid off', ((x0 + x1a) / 2, y2a / 2)),
          ('paid off', ((1 + x0) / 2, (y1b + 25) / 2)),
          ('paid off', ((1 + x2b) / 2, (y1b + 0) / 2)),
          ('paid off', ((x0 + x2b) / 2, (y1b + 0) / 2)),
         ]
for label, (x, y) in labels:
    ax.text(x, y, label, bbox={'facecolor':'white'},
            verticalalignment='center', horizontalalignment='center')

plt.tight_layout()
plt.show()


Gray lines show each split. The tree produces **piecewise axis-aligned** boundaries — unlike logistic regression's single diagonal line. This enables nonlinear, interaction-based patterns but cannot model diagonal boundaries efficiently.


### Measuring Impurity

**Gini:** $I = p(1-p)$ &emsp; **Entropy:** $I = -p\log_2 p - (1-p)\log_2(1-p)$

Both are zero for pure partitions and maximum at $p=0.5$. The best split minimizes:
$$I_{\text{split}} = \frac{|A_L|}{|A|}I(A_L) + \frac{|A_R|}{|A|}I(A_R)$$


In [ ]:
def entropyFunction(x):
    if x == 0: return 0
    return -x * math.log(x, 2) - (1 - x) * math.log(1 - x, 2)

def giniFunction(x):
    return x * (1 - x)


In [ ]:
x = np.linspace(0, 0.5, 50)
impure = pd.DataFrame({
    'x': x,
    'Accuracy': 2 * x,
    'Gini': [giniFunction(xi) / giniFunction(.5) for xi in x],
    'Entropy': [entropyFunction(xi) for xi in x],
})

fig, ax = plt.subplots(figsize=(5, 4))

impure.plot(x='x', y='Accuracy', ax=ax, linestyle='solid')
impure.plot(x='x', y='Entropy', ax=ax, linestyle='--')
impure.plot(x='x', y='Gini', ax=ax, linestyle=':')

ax.set_xlabel('Proportion p')
ax.set_ylabel('Impurity (normalized)')
ax.set_title('Impurity Measures: Gini vs. Entropy vs. Accuracy')
plt.tight_layout()
plt.show()


Gini and Entropy produce nearly identical trees in practice. Accuracy (misclassification rate) is a poor split criterion because it doesn't reward increasingly pure partitions.


## Bagging and the Random Forest

Random forests combine:
1. **Bagging:** Each tree trains on a bootstrap sample (with replacement).
2. **Feature subsampling:** Each split considers only $\sqrt{p}$ random features — reducing tree correlation.

Ensemble variance: $\rho\sigma^2 + \frac{(1-\rho)\sigma^2}{B}$. Feature subsampling reduces $\rho$, directly reducing overall variance.


In [ ]:
predictors = ['borrower_score', 'payment_inc_ratio']
outcome = 'outcome'

X = loan3000[predictors]
y = loan3000[outcome]

rf = RandomForestClassifier(n_estimators=500, random_state=1,
                            oob_score=True)
rf.fit(X, y)

oob = rf.oob_decision_function_
print("OOB decision function (first 3 and last 3 records):")
print(oob[:3])
print("...")
print(oob[-3:])
print(f"\nOOB accuracy: {rf.oob_score_:.4f}")
print(f"OOB error rate: {1 - rf.oob_score_:.4f}")


**OOB accuracy** is a free validation metric: each tree's ~37% left-out records serve as a built-in test set. No separate holdout needed.


In [ ]:
n_estimator = list(range(20, 510, 10))
oobScores = []
for n in n_estimator:
    rf = RandomForestClassifier(n_estimators=n,
                                criterion='entropy', max_depth=5,
                                random_state=1, oob_score=True)
    rf.fit(X, y)
    oobScores.append(rf.oob_score_)


In [ ]:
pd.DataFrame({
    'n': n_estimator,
    'oobScore': oobScores
}).plot(x='n', y='oobScore')
plt.xlabel('Number of Trees')
plt.ylabel('OOB Accuracy')
plt.title('Random Forest: OOB Accuracy vs. Number of Trees')
plt.show()


OOB accuracy improves quickly in the first 50–100 trees, then plateaus. More trees never hurt (unlike boosting), but returns diminish beyond ~200.


In [ ]:
predictions = X.copy()
predictions['prediction'] = rf.predict(X)
predictions.head()

fig, ax = plt.subplots(figsize=(5, 5))

predictions.loc[predictions.prediction=='paid off'].plot(
    x='borrower_score', y='payment_inc_ratio', style='.',
    markerfacecolor='none', markeredgecolor='C1', ax=ax)
predictions.loc[predictions.prediction=='default'].plot(
    x='borrower_score', y='payment_inc_ratio', style='o',
    markerfacecolor='none', markeredgecolor='C0', ax=ax)
ax.legend(['paid off', 'default']);
ax.set_xlim(0, 1)
ax.set_ylim(0, 25)
ax.set_xlabel('borrower_score')
ax.set_ylabel('payment_inc_ratio')
ax.set_title('Random Forest Predicted Outcomes')

plt.tight_layout()
plt.show()


The random forest produces a **complex, nonlinear** decision boundary vs. the single tree's axis-aligned rectangles. The trade-off: higher accuracy, but the 500-tree collective decision cannot be easily explained.


### Variable Importance

**Gini decrease:** Sum impurity reduction per feature across all trees (free, but biased toward high-cardinality features).

**Permutation accuracy decrease:** Shuffle each feature and measure accuracy loss — more reliable because it uses OOB data.


In [ ]:
predictors = ['loan_amnt', 'term', 'annual_inc', 'dti',
              'payment_inc_ratio', 'revol_bal', 'revol_util',
              'purpose', 'delinq_2yrs_zero', 'pub_rec_zero',
              'open_acc', 'grade', 'emp_length', 'purpose_',
              'home_', 'emp_len_', 'borrower_score']
outcome = 'outcome'

X = pd.get_dummies(loan_data[predictors], drop_first=True, dtype=int)
y = loan_data[outcome]

rf_all = RandomForestClassifier(n_estimators=500, random_state=1)
rf_all.fit(X, y)

rf_all_entropy = RandomForestClassifier(n_estimators=500, random_state=1,
                                        criterion='entropy')
print(rf_all_entropy.fit(X, y))


Two random forests are trained on the full feature set: one with **Gini** criterion (default) and one with **entropy**. Both use $500$ trees and all $45{,}342$ records. With $17+$ features expanded via one-hot encoding, the forest now has a rich set of predictors to work with.

In [ ]:
rf = RandomForestClassifier(n_estimators=100)
scores = defaultdict(list)

for _ in range(2):
    train_X, valid_X, train_y, valid_y = train_test_split(X, y,
                                                          test_size=0.3)
    rf.fit(train_X, train_y)
    acc = metrics.accuracy_score(valid_y, rf.predict(valid_X))
    for column in X.columns:
        X_t = valid_X.copy()
        X_t[column] = np.random.permutation(X_t[column].values)
        shuff_acc = metrics.accuracy_score(valid_y, rf.predict(X_t))
        scores[column].append((acc-shuff_acc)/acc)
print('Features sorted by their score:')
print(sorted([(round(np.mean(score), 4), feat) for
              feat, score in scores.items()], reverse=True))


`borrower_score` dominates (accuracy decrease ~0.07), far ahead of `grade` (~0.04). This validates our KNN feature engineering. Several features near zero importance could be dropped without loss.


In [ ]:
importances = rf_all.feature_importances_

df = pd.DataFrame({
    'feature': X.columns,
    'Accuracy decrease': [np.mean(scores[column]) for column in
                         X.columns],
    'Gini decrease': rf_all.feature_importances_,
    'Entropy decrease': rf_all_entropy.feature_importances_,
})
df = df.sort_values('Accuracy decrease')

fig, axes = plt.subplots(ncols=2, figsize=(10, 5))
ax = df.plot(kind='barh', x='feature', y='Accuracy decrease',
             legend=False, ax=axes[0])
ax.set_ylabel('')
ax.set_title('Accuracy Decrease (Permutation)')

ax = df.plot(kind='barh', x='feature', y='Gini decrease',
             legend=False, ax=axes[1])
ax.set_ylabel('')
ax.get_yaxis().set_visible(False)
ax.set_title('Gini Decrease (Impurity)')

plt.tight_layout()
plt.show()


**When the two measures disagree, trust permutation importance.** Gini importance is biased toward features used frequently but not necessarily usefully. `revol_bal` and `loan_amnt` show high Gini but near-zero permutation importance.


## Boosting

Boosting fits a **sequence** of weak models, each focusing on prior errors:
$$F(\mathbf{x}) = \alpha_1 f_1(\mathbf{x}) + \alpha_2 f_2(\mathbf{x}) + \cdots$$

**XGBoost** adds regularization to the loss:
$$\mathcal{L} = \sum_i \ell(y_i, \hat{y}_i) + \lambda\sum_j w_j^2 + \alpha\sum_j|w_j|$$


In [ ]:
predictors = ['borrower_score', 'payment_inc_ratio']
outcome = 'outcome'

X = loan3000[predictors]
y = pd.Series([1 if o == 'default' else 0 for o in loan3000[outcome]])

xgb = XGBClassifier(objective='binary:logistic', subsample=.63,
                    eval_metric='error')
print(xgb.fit(X, y))


`subsample=0.63` introduces stochasticity (stochastic gradient boosting). Shallow trees (default depth 6) are weak learners; sequential correction refines them over 100 rounds.


In [ ]:
xgb_df = X.copy()
xgb_df['prediction'] = ['default' if p == 1 else 'paid off' for p in xgb.predict(X)]
xgb_df['prob_default'] = xgb.predict_proba(X)[:, 0]
print(xgb_df.head())


The `prob_default` column shows model confidence per record. The 0.5 threshold converts probabilities to binary predictions.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

xgb_df.loc[xgb_df.prediction=='paid off'].plot(
    x='borrower_score', y='payment_inc_ratio', style='.',
    markerfacecolor='none', markeredgecolor='C1', ax=ax)
xgb_df.loc[xgb_df.prediction=='default'].plot(
    x='borrower_score', y='payment_inc_ratio', style='o',
    markerfacecolor='none', markeredgecolor='C0', ax=ax)
ax.legend(['paid off', 'default']);
ax.set_xlim(0, 1)
ax.set_ylim(0, 25)
ax.set_xlabel('borrower_score')
ax.set_ylabel('payment_inc_ratio')
ax.set_title('XGBoost Predicted Outcomes')

plt.tight_layout()
plt.show()


XGBoost and random forest produce qualitatively similar predictions. The key difference: random forest aggregates 500 independent trees; XGBoost builds 100 sequential, error-correcting trees.


### Regularization: Avoiding Overfitting

XGBoost can severely overfit without regularization — unlike random forests, which are robust to default parameters.


In [ ]:
predictors = ['loan_amnt', 'term', 'annual_inc', 'dti',
              'payment_inc_ratio', 'revol_bal', 'revol_util',
              'purpose', 'delinq_2yrs_zero', 'pub_rec_zero',
              'open_acc', 'grade', 'emp_length', 'purpose_',
              'home_', 'emp_len_', 'borrower_score']
outcome = 'outcome'

X = pd.get_dummies(loan_data[predictors], drop_first=True, dtype=int)
y = pd.Series([1 if o == 'default' else 0 for o in loan_data[outcome]])

train_X, valid_X, train_y, valid_y = train_test_split(X, y, test_size=10000)

xgb_default = XGBClassifier(objective='binary:logistic', n_estimators=250, max_depth=6,
                            reg_lambda=0, learning_rate=0.3, subsample=1,
                            eval_metric='error')
xgb_default.fit(train_X, train_y)

xgb_penalty = XGBClassifier(objective='binary:logistic', n_estimators=250, max_depth=6,
                            reg_lambda=1000, learning_rate=0.1, subsample=0.63,
                            eval_metric='error')
print(xgb_penalty.fit(train_X, train_y))


Two models on the same data: **Default** (`reg_lambda=0`, `lr=0.3`, full sample) vs. **Regularized** (`reg_lambda=1000`, `lr=0.1`, 63% subsample).


In [ ]:
pred_default = xgb_default.predict_proba(train_X)[:, 1]
error_default = abs(train_y - pred_default) > 0.5
print('default (train): ', np.mean(error_default))

pred_default = xgb_default.predict_proba(valid_X)[:, 1]
error_default = abs(valid_y - pred_default) > 0.5
print('default: ', np.mean(error_default))

pred_penalty = xgb_penalty.predict_proba(valid_X)[:, 1]
error_penalty = abs(valid_y - pred_penalty) > 0.5
print('penalty: ', np.mean(error_penalty))


**Overfitting evidence:** Default model — training error ~10%, test error ~36% (>20pp gap). Regularized model — test error ~33%, with a much smaller train-test gap. More regularization → better generalization.


In [ ]:
results = []
for ntree_limit in list(range(1, 20)) + list(range(20, 250, 5)):
    iteration_range = [1, ntree_limit + 1]
    train_default = xgb_default.predict_proba(train_X, iteration_range=iteration_range)[:, 1]
    train_penalty = xgb_penalty.predict_proba(train_X, iteration_range=iteration_range)[:, 1]
    pred_default = xgb_default.predict_proba(valid_X, iteration_range=iteration_range)[:, 1]
    pred_penalty = xgb_penalty.predict_proba(valid_X, iteration_range=iteration_range)[:, 1]
    results.append({
        'iterations': ntree_limit,
        'default train': np.mean(abs(train_y - train_default) > 0.5),
        'penalty train': np.mean(abs(train_y - train_penalty) > 0.5),
        'default test': np.mean(abs(valid_y - pred_default) > 0.5),
        'penalty test': np.mean(abs(valid_y - pred_penalty) > 0.5),
    })

results = pd.DataFrame(results)
print(results.head())


First few iterations: both models improve rapidly. The train-test gap starts small but widens dramatically for the default model as boosting rounds increase.


In [ ]:
ax = results.plot(x='iterations', y='default test')
results.plot(x='iterations', y='penalty test', ax=ax)
results.plot(x='iterations', y='default train', ax=ax)
results.plot(x='iterations', y='penalty train', ax=ax)
ax.set_xlabel('Number of Boosting Rounds')
ax.set_ylabel('Error Rate')
ax.set_title('Default vs. Regularized XGBoost')
plt.tight_layout()
plt.show()


This plot shows the full overfitting story: the default model's training error plummets while test error stalls; the regularized model's training and test errors track together. **Never deploy XGBoost without regularization.**


### Hyperparameters and Cross-Validation

Key XGBoost hyperparameters are `learning_rate` (eta) and `max_depth`. Use $k$-fold cross-validation to find the optimal combination.


In [ ]:
idx = np.random.choice(range(5), size=len(X), replace=True)
error = []
for eta, max_depth in product([0.1, 0.5, 0.9], [3, 6, 9]):
    xgb = XGBClassifier(objective='binary:logistic', n_estimators=100,
                        max_depth=max_depth, learning_rate=eta,
                        eval_metric='error')
    cv_error = []
    for k in range(5):
        fold_idx = idx == k
        train_X = X.loc[~fold_idx]; train_y = y[~fold_idx]
        valid_X = X.loc[fold_idx]; valid_y = y[fold_idx]

        xgb.fit(train_X, train_y)
        pred = xgb.predict_proba(valid_X)[:, 1]
        cv_error.append(np.mean(abs(valid_y - pred) > 0.5))
    error.append({
        'eta': eta,
        'max_depth': max_depth,
        'avg_error': np.mean(cv_error)
    })
    print(error[-1])
errors = pd.DataFrame(error)


5-fold CV across 9 combinations of `eta` ∈ {0.1, 0.5, 0.9} and `max_depth` ∈ {3, 6, 9}:


In [ ]:
print(errors.pivot_table(index='eta', columns='max_depth', values='avg_error') * 100)
